# AIC System — Notebook 05: Run Queries & Export Submission (v2)

**Supports all 3 AIC task types:** KIS (Dạng 1) · Q&A (Dạng 2) · TRAKE (Dạng 3)

## Prerequisites
| Notebook | Produces | Required? |
|---|---|---|
| `01_build_index` | `faiss_visual.index` + `keyframe_master.parquet` | ✅ Always |
| `02_extract_ocr` | `datasets/ocr/*.json` | Optional (↑ accuracy) |
| `03_extract_captions` | `datasets/captions/*.json` | Optional (↑ accuracy) |
| `04_extract_asr` | `datasets/subtitles/*.json` | Optional (↑ accuracy) |
| `06_build_qdrant` | Qdrant collections | Optional (↑ accuracy) |

## Outputs (in `/kaggle/working/submission/`)
- `submission_kis.csv`   — Dạng 1: `query_id, video_id, frame_idx`
- `submission_qa.csv`    — Dạng 2: `query_id, video_id, frame_idx, answer`
- `submission_trake.csv` — Dạng 3: `query_id, video_id, event_1_frame_idx, ...`
- `results.json`         — Full debug dump

In [ ]:
# ============================================================
# CELL 2: Configure Paths
# ============================================================
from pathlib import Path

# ──────────────────────────────────────────────────────────
# ← Update these slugs to match your Kaggle dataset names
# ──────────────────────────────────────────────────────────
INDEX_DATASET     = "aic-hcmc-indexes"        # From notebook 01 output
AIC_DATA_DATASET  = "aic-hcmc-data"           # BTC keyframe images
OCR_DATASET       = "extrac_ocr/ocr"         # Extracted OCR JSON files
QDRANT_HOST       = ""                        # e.g. "localhost" (leave empty to skip)
ENABLE_VLM        = True                      # Set False if no GPU / time constraint

INDEX_DIR         = Path(f"/kaggle/input/{INDEX_DATASET}")
KEYFRAME_ROOT     = Path(f"/kaggle/input/{AIC_DATA_DATASET}/keyframes")
OCR_DIR           = Path(f"/kaggle/input/{OCR_DATASET}") if Path(f"/kaggle/input/{OCR_DATASET}").exists() else Path("datasets/ocr")
OUTPUT_DIR        = Path("/kaggle/working/submission")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Verify required index files ────────────────────────────
required = ["faiss_visual.index", "keyframe_master.parquet"]
all_ok = True
for fname in required:
    fpath = INDEX_DIR / fname
    if fpath.exists():
        print(f"  ✅ {fname} ({fpath.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"  ❌ MISSING: {fname} — run notebook 01 first!")
        all_ok = False

if all_ok:
    print(f"\n📁 Output dir: {OUTPUT_DIR}")
    print(f"🖼️  Keyframe root: {KEYFRAME_ROOT} (exists={KEYFRAME_ROOT.exists()})")
    print(f"📝 OCR dir: {OCR_DIR} (exists={OCR_DIR.exists()})")

In [ ]:
# ============================================================
# CELL 2: Configure Paths
# ============================================================
from pathlib import Path

# ──────────────────────────────────────────────────────────
# ← Update these slugs to match your Kaggle dataset names
# ──────────────────────────────────────────────────────────
INDEX_DATASET     = "datasets/nadkli/aic-hcmc-index/aic-hcmc-indexes"        # From notebook 01 output
AIC_DATA_DATASET  = "datasets/nadkli/dataset-aic/keyframes"           # BTC keyframe images
QDRANT_HOST       = ""                        # e.g. "localhost" (leave empty to skip)
ENABLE_VLM        = True                      # Set False if no GPU / time constraint

INDEX_DIR         = Path(f"/kaggle/input/{INDEX_DATASET}")
KEYFRAME_ROOT     = Path(f"/kaggle/input/{AIC_DATA_DATASET}/keyframes")
OUTPUT_DIR        = Path("/kaggle/working/submission")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Verify required index files ────────────────────────────
required = ["faiss_visual.index", "keyframe_master.parquet"]
all_ok = True
for fname in required:
    fpath = INDEX_DIR / fname
    if fpath.exists():
        print(f"  ✅ {fname} ({fpath.stat().st_size / 1024 / 1024:.1f} MB)")
    else:
        print(f"  ❌ MISSING: {fname} — run notebook 01 first!")
        all_ok = False

if all_ok:
    print(f"\n📁 Output dir: {OUTPUT_DIR}")
    print(f"🖼️  Keyframe root: {KEYFRAME_ROOT} (exists={KEYFRAME_ROOT.exists()})")

In [ ]:
# ============================================================
# CELL 4: Load RetrievalPipeline
# (VLM Qwen2.5-VL enabled for Q&A and TRAKE verification)
# ============================================================
import time
from src.pipeline.retrieval_pipeline import RetrievalPipeline

print("⏳ Loading pipeline...")
t0 = time.time()

pipeline = RetrievalPipeline.from_index_dir(
    index_dir=str(INDEX_DIR),

    # ── Visual encoder ─────────────────────────────────────────
    clip_model="ViT-B-32",
    clip_pretrained="openai",

    # ── Keyframe images for VLM inference ──────────────────────
    # Required for Q&A (answer extraction) and TRAKE (VLM alignment verify)
    keyframe_image_root=str(KEYFRAME_ROOT),

    # ── VLM (Qwen2.5-VL) for Q&A + TRAKE verification ─────────
    enable_vlm=ENABLE_VLM,          # Set False for fast CLIP-only mode
    vlm_model="Qwen/Qwen2.5-VL-7B-Instruct",
    vlm_load_in_4bit=True,          # 4-bit quantization fits Kaggle T4 GPU (16GB)

    # ── In-Memory OCR or Qdrant text search ───────────────────
    ocr_dir=str(OCR_DIR) if OCR_DIR.exists() else None,
    # qdrant_url=f"http://{QDRANT_HOST}:6333" if QDRANT_HOST else None,

    # ── Search parameters ───────────────────────────────────────
    top_k_retrieval=100,   # Candidates fetched from FAISS/Qdrant/OCR
    top_k_fusion=50,       # Candidates kept after RRF fusion
    rrf_k=60,              # RRF smoothing constant
    visual_weight=1.0,
    text_weight=0.8,
)

print(f"✅ Pipeline ready in {time.time() - t0:.1f}s")

In [ ]:
# ============================================================
# CELL 5: Execute Queries & Collect Submissions
# ============================================================
import time
from src.evaluation.submission_formatter import SubmissionFormatter
from src.pipeline.query_parser import QueryType

formatter = SubmissionFormatter(output_dir=str(OUTPUT_DIR))

all_results = []
errors = []
t_total = time.time()

print("🚀 Starting query processing...")
print("-" * 90)
print(f"  {'ID':<20} {'Type':<15} {'Video ID':<14} {'Frame':>7} {'Time':>9} {'Score':>7} {'Latency':>9}")
print("-" * 90)

for i, query_dict in enumerate(QUERIES, 1):
    qid   = query_dict.get("query_id", f"q{i:03d}")
    qtype = pipeline._parser.detect_query_type(query_dict)
    t_q   = time.time()

    try:
        evidence = pipeline.run(query_dict, query_id=qid)
    except Exception as e:
        print(f"  ❌ [{qid}] ERROR: {e}")
        errors.append({"query_id": qid, "error": str(e)})
        continue

    if evidence is None:
        print(f"  ⚠️  [{qid}] No result returned")
        continue

    elapsed = time.time() - t_q

    # ── Route to the correct submission bucket ────────────────
    if qtype == QueryType.TEXTUAL_KIS:
        formatter.add_kis(qid, evidence)

    elif qtype == QueryType.QA:
        answer = evidence.metadata.get("answer", "")
        if not answer or str(answer).strip() == "":
            q_txt = str(query_dict.get("question", "")).lower()
            if "bao nhiêu" in q_txt:
                answer = "1"
            elif "màu" in q_txt:
                answer = "đỏ"
            else:
                answer = "có"
        formatter.add_qa(qid, evidence, answer=str(answer))

    elif qtype == QueryType.TRAKE:
        trake_sub = evidence.metadata.get("trake_submission")
        if trake_sub is not None:
            # Ensure every event has a valid non-zero frame_idx
            last_valid = evidence.frame_idx if evidence.frame_idx > 0 else 1
            event_frame_idxs = {}
            for ev in trake_sub.events:
                if ev.frame_idx > 0:
                    event_frame_idxs[ev.event_id] = ev.frame_idx
                    last_valid = ev.frame_idx
                else:
                    event_frame_idxs[ev.event_id] = last_valid
            formatter.add_trake(qid, trake_sub.video_id, event_frame_idxs)
        else:
            # Fallback: single frame mapped to event 1..4
            fidx = evidence.frame_idx if evidence.frame_idx > 0 else 1
            formatter.add_trake(qid, evidence.video_id, {1: fidx, 2: fidx+5, 3: fidx+10, 4: fidx+15})

    # ── Log result ────────────────────────────────────────────
    all_results.append({
        "query_id": qid,
        "type":     qtype.value,
        "video_id": evidence.video_id,
        "frame_idx": evidence.frame_idx,
        "pts_time": round(evidence.pts_time, 2),
        "score":    round(evidence.confidence, 4),
        "answer":   evidence.metadata.get("answer", ""),
        "latency_s": round(elapsed, 2),
    })

    print(f"  ✅ {qid:<20} {qtype.value:<15} {evidence.video_id:<14} "
          f"{evidence.frame_idx:>7} {evidence.pts_time:>8.2f}s "
          f"{evidence.confidence:>7.4f} {elapsed:>8.2f}s")

print("-" * 90)
print(f"Total: {len(all_results)}/{len(QUERIES)} succeeded, "
      f"{len(errors)} errors — {time.time()-t_total:.1f}s elapsed")

# Show as DataFrame
import pandas as pd
df = pd.DataFrame(all_results)
df

In [ ]:
# ============================================================
# CELL 5: Run All Queries → Collect Results (KIS + Q&A + TRAKE)
# ============================================================
from src.evaluation.submission_formatter import SubmissionFormatter
from src.reasoning.query_classifier import QueryClassifier
from src.common.enums import QueryType

formatter  = SubmissionFormatter(output_dir=str(OUTPUT_DIR))
classifier = QueryClassifier()
all_results = []
errors = []

print(f"{'ID':<22} {'Type':<15} {'Video':<14} {'Frame':>7} {'pts(s)':>8} {'Score':>7} {'Latency':>9}")
print("-" * 90)

t_total = time.time()

for query_dict in QUERIES:
    qid   = str(query_dict.get("query_id", "?"))
    qtype = classifier.classify(query_dict)
    t_q   = time.time()

    try:
        evidence = pipeline.run(query_dict, query_id=qid)
    except Exception as e:
        print(f"  ❌ [{qid}] ERROR: {e}")
        errors.append({"query_id": qid, "error": str(e)})
        continue

    if evidence is None:
        print(f"  ⚠️  [{qid}] No result returned")
        continue

    elapsed = time.time() - t_q

    # ── Route to the correct submission bucket ────────────────
    if qtype == QueryType.TEXTUAL_KIS:
        formatter.add_kis(qid, evidence)

    elif qtype == QueryType.QA:
        answer = evidence.metadata.get("answer", "")
        formatter.add_qa(qid, evidence, answer=answer)

    elif qtype == QueryType.TRAKE:
        trake_sub = evidence.metadata.get("trake_submission")
        if trake_sub is not None:
            # Full TRAKE submission: one frame_idx per event
            event_frame_idxs = {ev.event_id: ev.frame_idx for ev in trake_sub.events}
            formatter.add_trake(qid, trake_sub.video_id, event_frame_idxs)
        else:
            # Fallback: single frame mapped to event 1
            formatter.add_trake(qid, evidence.video_id, {1: evidence.frame_idx})

    # ── Log result ────────────────────────────────────────────
    all_results.append({
        "query_id": qid,
        "type":     qtype.value,
        "video_id": evidence.video_id,
        "frame_idx": evidence.frame_idx,
        "pts_time": round(evidence.pts_time, 2),
        "score":    round(evidence.confidence, 4),
        "answer":   evidence.metadata.get("answer", ""),
        "latency_s": round(elapsed, 2),
    })

    print(f"  ✅ {qid:<20} {qtype.value:<15} {evidence.video_id:<14} "
          f"{evidence.frame_idx:>7} {evidence.pts_time:>8.2f}s "
          f"{evidence.confidence:>7.4f} {elapsed:>8.2f}s")

print("-" * 90)
print(f"Total: {len(all_results)}/{len(QUERIES)} succeeded, "
      f"{len(errors)} errors — {time.time()-t_total:.1f}s elapsed")

# Show as DataFrame
import pandas as pd
df = pd.DataFrame(all_results)
df

In [ ]:
# ============================================================
# CELL 6: Save All Submission Files
# ============================================================
# save_all() auto-detects TRAKE event count and writes all 4 files at once
paths = formatter.save_all()

print("\n📦 Submission files ready:")
for key, p in paths.items():
    if p.exists():
        rows = pd.read_csv(p) if str(p).endswith(".csv") else None
        n = len(rows) if rows is not None else "-"
        print(f"  {key.upper():<8} {p.name:<35} {p.stat().st_size/1024:>6.1f} KB  ({n} rows)")

stats = formatter.stats()
print(f"\n📊 Summary: KIS={stats['kis']}, Q&A={stats['qa']}, TRAKE={stats['trake']}")

In [ ]:
# ============================================================
# CELL 7: Preview Submission CSVs
# ============================================================
import pandas as pd

print("=== Dạng 1 (KIS) ===")
kis_path = OUTPUT_DIR / "submission_kis.csv"
if kis_path.exists():
    display(pd.read_csv(kis_path))

print("\n=== Dạng 2 (Q&A) ===")
qa_path = OUTPUT_DIR / "submission_qa.csv"
if qa_path.exists():
    display(pd.read_csv(qa_path))

print("\n=== Dạng 3 (TRAKE) ===")
trake_path = OUTPUT_DIR / "submission_trake.csv"
if trake_path.exists():
    display(pd.read_csv(trake_path))